# Task 3: Linear Regression

**AI & ML Internship**

### Objective
Implement and understand simple and multiple linear regression using the provided `Housing.csv` dataset.

### Tools
- Python
- Pandas
- NumPy
- Matplotlib
- Scikit-learn

### This notebook covers
- Dataset loading and inspection
- Data preprocessing
- Train-test split
- Simple Linear Regression
- Multiple Linear Regression
- MAE, MSE and R² evaluation
- Regression line visualization
- Coefficient interpretation

**Note:** The interview questions from the task sheet are intentionally skipped.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from google.colab import files


In [ ]:
# Upload Housing.csv in Google Colab
uploaded = files.upload()

file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
df.head()


In [ ]:
# Inspect the dataset
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
# Basic descriptive statistics
df.describe(include="all").T


## Data Preprocessing

The target variable is `price`. Categorical columns are converted into numerical dummy variables using one-hot encoding. This makes the dataset suitable for scikit-learn's Linear Regression model.

In [ ]:
# Remove duplicate rows if any
df = df.drop_duplicates().copy()

# Handle missing values
# Numeric columns: fill with median
numeric_cols = df.select_dtypes(include=np.number).columns
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Categorical columns: fill with mode
categorical_cols = df.select_dtypes(exclude=np.number).columns
for col in categorical_cols:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].mode()[0])

print("Missing values after preprocessing:")
print(df.isnull().sum())


In [ ]:
# Separate target and features
target = "price"

X = df.drop(columns=[target])
y = df[target]

# Convert categorical features to dummy variables
X_encoded = pd.get_dummies(X, drop_first=True, dtype=int)

print("Original feature count:", X.shape[1])
print("Encoded feature count:", X_encoded.shape[1])

X_encoded.head()


## Simple Linear Regression

For simple linear regression, `area` is used as the single predictor and `price` is the target.

In [ ]:
# Prepare simple regression data
X_simple = df[["area"]]
y_simple = df["price"]

X_train_simple, X_test_simple, y_train_simple, y_test_simple = train_test_split(
    X_simple, y_simple, test_size=0.2, random_state=42
)

# Train simple linear regression model
simple_model = LinearRegression()
simple_model.fit(X_train_simple, y_train_simple)

# Predictions
y_pred_simple = simple_model.predict(X_test_simple)

# Evaluation
mae_simple = mean_absolute_error(y_test_simple, y_pred_simple)
mse_simple = mean_squared_error(y_test_simple, y_pred_simple)
r2_simple = r2_score(y_test_simple, y_pred_simple)

print("Simple Linear Regression Results")
print("-" * 35)
print(f"MAE: {mae_simple:,.2f}")
print(f"MSE: {mse_simple:,.2f}")
print(f"R² Score: {r2_simple:.4f}")

print(f"\nIntercept: {simple_model.intercept_:,.2f}")
print(f"Area coefficient: {simple_model.coef_[0]:,.2f}")


In [ ]:
# Plot simple linear regression line
plt.figure(figsize=(9, 6))

plt.scatter(X_test_simple["area"], y_test_simple, alpha=0.7, label="Actual values")

# Sort x values so the regression line is displayed correctly
sort_idx = np.argsort(X_test_simple["area"].values)
x_sorted = X_test_simple["area"].values[sort_idx]
y_line = simple_model.predict(x_sorted.reshape(-1, 1))

plt.plot(x_sorted, y_line, linewidth=2, label="Regression line")

plt.xlabel("Area")
plt.ylabel("Price")
plt.title("Simple Linear Regression: Area vs Price")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


### Simple Regression Interpretation

- The **intercept** is the predicted house price when the area is zero; it mainly serves as the mathematical starting point of the regression equation.
- The **area coefficient** represents the estimated change in house price for a one-unit increase in area, while keeping the simple model's structure unchanged.
- The **R² score** indicates how much of the variation in house prices is explained by area alone.

## Multiple Linear Regression

Multiple linear regression uses all available features after categorical variables have been converted into dummy variables.

In [ ]:
# Split encoded data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])


In [ ]:
# Train multiple linear regression model
multiple_model = LinearRegression()
multiple_model.fit(X_train, y_train)

# Predictions
y_pred = multiple_model.predict(X_test)

# Evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Multiple Linear Regression Results")
print("-" * 40)
print(f"MAE: {mae:,.2f}")
print(f"MSE: {mse:,.2f}")
print(f"R² Score: {r2:.4f}")
print(f"Intercept: {multiple_model.intercept_:,.2f}")


In [ ]:
# Compare actual and predicted prices
comparison = pd.DataFrame({
    "Actual Price": y_test.values,
    "Predicted Price": y_pred
})

comparison["Absolute Error"] = abs(
    comparison["Actual Price"] - comparison["Predicted Price"]
)

comparison.head(10)


In [ ]:
# Plot actual vs predicted prices
plt.figure(figsize=(9, 6))

plt.scatter(y_test, y_pred, alpha=0.7)

min_value = min(y_test.min(), y_pred.min())
max_value = max(y_test.max(), y_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    linewidth=2,
    label="Perfect prediction"
)

plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Multiple Linear Regression: Actual vs Predicted Prices")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()


## Model Coefficients

The coefficients show the estimated contribution of each feature to the predicted house price, while the other included features are held constant.

For categorical variables, each coefficient is interpreted relative to the category that was dropped during one-hot encoding.

In [ ]:
# Display coefficients
coefficients = pd.DataFrame({
    "Feature": X_encoded.columns,
    "Coefficient": multiple_model.coef_
})

coefficients["Absolute Coefficient"] = coefficients["Coefficient"].abs()
coefficients = coefficients.sort_values("Absolute Coefficient", ascending=False)

coefficients


In [ ]:
# Visualize the most influential coefficients
top_n = min(10, len(coefficients))
top_coefficients = coefficients.head(top_n).sort_values("Coefficient")

plt.figure(figsize=(10, 6))
plt.barh(top_coefficients["Feature"], top_coefficients["Coefficient"])
plt.xlabel("Coefficient")
plt.ylabel("Feature")
plt.title(f"Top {top_n} Linear Regression Coefficients")
plt.grid(True, axis="x", alpha=0.3)
plt.show()


## Final Model Summary

### Simple Linear Regression
Uses only **area** to predict house price.

### Multiple Linear Regression
Uses all available numerical and encoded categorical features to predict house price.

### Evaluation Metrics
- **MAE (Mean Absolute Error):** average absolute difference between actual and predicted prices.
- **MSE (Mean Squared Error):** average squared prediction error; larger errors have a stronger effect.
- **R² Score:** measures the proportion of target variation explained by the model.

The model with the more useful combination of error metrics and R² should be preferred for this dataset.

In [ ]:
# Final side-by-side metric comparison
results = pd.DataFrame({
    "Model": ["Simple Linear Regression", "Multiple Linear Regression"],
    "MAE": [mae_simple, mae],
    "MSE": [mse_simple, mse],
    "R2": [r2_simple, r2]
})

results
